# Phase 3, step 4 — knowing when to stop

The step-3 notebook ended with a live run that worked. It also ended with a
problem, visible in its own leaderboard: **eleven experiments, and the last six
moved the score by about 0.003.**

Nothing was broken. The agent had tools, the tools worked, every experiment was
recorded. It simply had no notion of *done*, so it kept going — and would have
kept going until the context ran out, spending real money to re-roll a die.

This notebook is about handing that judgement to the harness. It is a smaller
slice than the previous ones, and the interesting part is not the code. It is
the two questions underneath:

1. What counts as an improvement, when every number is noisy?
2. Who gets to say the run is over — the model, or the harness?

## 1. The evidence

Here is the real leaderboard from the step-3 run, typed out. Look at it the way
you would look at someone else's experiment log.

In [1]:
import json
import pandas as pd

def exp(id, model_type, cv_score, fold_spread, source_table):
    return {"id": id, "model_type": model_type, "cv_score": cv_score,
            "fold_spread": fold_spread, "source_table": source_table}

observed = [
    exp("e1",  "baseline", 0.6903, 0.0004, "apartments.csv"),
    exp("e2",  "lgbm",     0.5049, 0.0374, "tables/t2.parquet"),
    exp("e3",  "xgb",      0.3694, 0.0301, "tables/t2.parquet"),
    exp("e4",  "catboost", 0.2744, 0.0209, "tables/t2.parquet"),
    exp("e5",  "catboost", 0.2779, 0.0221, "tables/t4.parquet"),
    exp("e6",  "catboost", 0.2987, 0.0244, "tables/t2.parquet"),
    exp("e7",  "catboost", 0.2955, 0.0233, "tables/t4.parquet"),
    exp("e8",  "xgb",      0.2741, 0.0252, "tables/t2.parquet"),
    exp("e9",  "xgb",      0.2718, 0.0247, "tables/t4.parquet"),
    exp("e10", "lgbm",     0.3130, 0.0288, "tables/t4.parquet"),
    exp("e11", "catboost", 0.2779, 0.0231, "tables/t4.parquet"),
]
pd.DataFrame(observed)

,id,model_type,cv_score,fold_spread,source_table
0,e1,baseline,0.6903,0.0004,apartments.csv
1,e2,lgbm,0.5049,0.0374,tables/t2.parquet
2,e3,xgb,0.3694,0.0301,tables/t2.parquet
3,e4,catboost,0.2744,0.0209,tables/t2.parquet
4,e5,catboost,0.2779,0.0221,tables/t4.parquet
5,e6,catboost,0.2987,0.0244,tables/t2.parquet
6,e7,catboost,0.2955,0.0233,tables/t4.parquet
7,e8,xgb,0.2741,0.0252,tables/t2.parquet
8,e9,xgb,0.2718,0.0247,tables/t4.parquet
9,e10,lgbm,0.3130,0.0288,tables/t4.parquet


The run really got somewhere: `0.6903` to `0.2744` in four experiments. Then
seven more experiments bought `0.2744` to `0.2718` — a gain of `0.0026`, while
each model's own score wobbles by `±0.02` between folds.

That last comparison is the whole notebook. The improvement is **an order of
magnitude smaller than the noise in the measurement it is made of**. If you
re-ran e9 with a different fold seed it could easily come out worse than e4.

**Check yourself:** e9 really does have the lowest number. What is wrong with
shipping it?

> Nothing, exactly — it is a fine model, and `finalize_run` will happily package
> it. What is wrong is *believing* it is better than e4. Picking the minimum of
> eleven noisy measurements gives you a number biased low: you have selected for
> lucky folds as much as for a good model. The cost is not a bad deliverable, it
> is seven experiments spent buying a difference you cannot measure.

## 2. What counts as an improvement

The tempting rule is "stop when the gain is under 0.001". That needs a different
number for every metric and every dataset, and getting it wrong is silent — too
strict and you stop mid-climb, too loose and you never stop.

The yardstick is already in the leaderboard. `fold_spread` is the standard
deviation of the incumbent's score across folds: how much this exact model's
number moves when you merely change which rows it trained on. A gain smaller
than that is not a smaller gain, it is the same measurement taken twice.

So: **a gain counts if it exceeds `min_gain_ratio` x the incumbent's fold
spread.** Everything else is a tie.

In [2]:
from tabular_agent.harness.stop import StopPolicy, evaluate, advisory

policy = StopPolicy()
print(policy)

status = evaluate(observed, "logloss", policy)
print(json.dumps(status, indent=2))

StopPolicy(patience=10, min_gain_ratio=0.25, table_patience=3, max_experiments=25, max_refusals=3)
{
  "should_stop": false,
  "reason": null,
  "n_experiments": 11,
  "since_improvement": 7,
  "patience": 10,
  "experiments_remaining": 14
}


`since_improvement: 7` — seven experiments since anything real happened.

And yet `should_stop` is **False**, because `patience` is 10. That is not an
oversight, and it is the most important thing in this notebook: seven ideas that
did not work is not, on its own, a reason to end a search. Section 4 is about
what *is*.

Note also what the machinery does not claim. It says nothing about the model
being bad or the data being exhausted — only that the last seven experiments
produced no measurable improvement. That is a statement about the search.

## 3. Patience, and why it is not 1

Stopping at the first non-improvement would be wrong, because a run **branches**.
The agent builds a feature table, tries something on top that turns out worse,
backs up, tries a different encoder. A dip is the search working.

In [3]:
branching = [
    {"id": "e1", "model_type": "lgbm", "cv_score": 0.40, "fold_spread": 0.02},
    {"id": "e2", "model_type": "lgbm", "cv_score": 0.55, "fold_spread": 0.03},   # worse
    {"id": "e3", "model_type": "lgbm", "cv_score": 0.52, "fold_spread": 0.03},   # still worse
    {"id": "e4", "model_type": "lgbm", "cv_score": 0.31, "fold_spread": 0.02},   # the payoff
]
for k in ("since_improvement", "should_stop"):
    print(k, "=", evaluate(branching, "logloss", policy)[k])

since_improvement = 0
should_stop = False


Two bad experiments in a row, and the run survives to find the good one. This is
early stopping, with `patience` meaning exactly what it means in boosting — and
inheriting the same trade-off: too little and you stop on a plateau that had
something behind it, too much and you pay for experiments you already know are
going nowhere.

**Check yourself:** the incumbent in that example stays `0.40` through e2 and e3,
even though those are the most recent results. Why not compare each experiment to
the one before it?

> Because "better than the last thing I tried" is a ratchet that never stops. A
> run alternating `0.40, 0.55, 0.41, 0.56, 0.42` would report an improvement every
> other step while going nowhere. The incumbent is the *best so far*, so beating it
> means beating everything — which is what "progress" has to mean.

## 4. Not all misses are equal

Here is where the first version of this rule was wrong, and it is worth seeing
the mistake because it is an easy one to make.

Patience alone treats these two runs as identical — same number of experiments,
same absence of improvement, same `since_improvement`:

In [4]:
tuning = [
    {"id": "e1", "model_type": "lgbm", "cv_score": 0.40, "fold_spread": 0.02,
     "source_table": "tables/t1.parquet"},
]
for i, s in enumerate([0.41, 0.42, 0.415, 0.43, 0.44]):
    tuning.append({"id": f"e{i + 2}", "model_type": "lgbm", "cv_score": s,
                   "fold_spread": 0.02, "source_table": "tables/t1.parquet"})

searching = [dict(r, source_table=f"tables/t{i + 1}.parquet")
             for i, r in enumerate(tuning)]

for name, lb in [("tuning", tuning), ("searching", searching)]:
    st = evaluate(lb, "logloss", policy)
    print(f"{name:10} since_improvement={st['since_improvement']}  should_stop={st['should_stop']}")

tuning     since_improvement=5  should_stop=False
searching  since_improvement=5  should_stop=False


But they are not the same run at all.

`searching` tried **six different feature tables**. Six ideas, none of which
worked. That is not a failing search — it is what search *is*, and a rule that
cuts it off is worse than no rule. In ML you try a lot of things that go nowhere
before one lands.

`tuning` trained on **one table six times**. The only things that could have
differed are the model and its hyper-parameters. That is not six experiments; it
is one experiment costed six times. The Kaggle-consensus version of this: feature
work and a trustworthy validation setup are where the movement is, and tuning is
worth a small fraction of it — real, but small, and best done *once at the end*
on the feature set you've settled on, not interleaved with the search.

So the run-level budget is **generous** (`patience=10`), and the sharp constraint
is **per table**:

In [5]:
from tabular_agent.harness.stop import table_status

print("tuning, t1     :", table_status(tuning, "logloss", policy, "tables/t1.parquet")["exhausted"])
print("searching, t1  :", table_status(searching, "logloss", policy, "tables/t1.parquet")["exhausted"])
print()
print(table_status(tuning, "logloss", policy, "tables/t1.parquet")["reason"])

tuning, t1     : True
searching, t1  : False

5 experiments on tables/t1.parquet in a row produced no real improvement, so it is only the model and its hyper-parameters changing now. That is the cheapest experiment available and the least informative one


Now run it against the real leaderboard from section 1 — the one the run-level
rule was content to let continue.

In [6]:
for t in ("tables/t2.parquet", "tables/t4.parquet"):
    st = table_status(observed, "logloss", policy, t)
    print(f"{t}: {st['n_experiments_on_table']} experiments, "
          f"{st['since_improvement_on_table']} since an improvement -> "
          f"exhausted={st['exhausted']}")

tables/t2.parquet: 5 experiments, 2 since an improvement -> exhausted=False
tables/t4.parquet: 5 experiments, 2 since an improvement -> exhausted=False


Two of three on each — **not** exhausted. Worth sitting with, because it is the
honest result and it is not the flattering one.

That run alternated: `t2, t4, t2, t4, t2, t4, t4`. Changing the model each time
and switching tables every other experiment, it kept both per-table counters
below the limit while doing exactly the thing the rule exists to catch.
Alternating between two dead tables evades a per-table counter, by construction.

So what actually bounds that run is the pair of rules together: it sits at
`since_improvement=7` of 10 at e11, and two or three more empty experiments end
it at the run level. Which means — say it plainly — **on this particular run the
new rule is more permissive than the old one**: ~14 experiments where `patience=4`
stopped at 8.

That is the trade being made, not an accident. A rule tight enough to catch this
run at e8 also stops the run that spends eight experiments on eight real feature
ideas before the ninth lands, and that second failure is much more expensive: one
wastes money, the other throws away the answer.

**Check yourself:** could you close the alternation loophole by counting
"experiments since improvement on *any* table the run has already exhausted"?

> You could, and it would catch this run. It would also mean a table's status
> depends on what happened on other tables, so a genuinely promising `t5` could
> arrive already condemned by the history of `t2` and `t4`. Every tightening buys
> one failure and sells another; the question is only which one you would rather
> explain.

And critically, an exhausted table is refused — **the run is not**. The agent is
told to go build features and carries straight on. A table is cheap to give up
on; a run is not. Only one of those deserves a hair trigger.

That also settles a question left open in step 3: whether to put "prefer feature
work over hyperparameter tuning" in the system prompt. It does not need to be
there. It is now a property of the tool, where it cannot be ignored, forgotten,
or argued with — the same move as every other guarantee in this harness.

**Check yourself:** the per-table rule counts experiments on a table since *that
table's* best. Why not since the *run's* best?

> Because a table that never beat the global best could still be improving. You
> build `t3`, it scores worse than `t2` but each experiment on it is getting
> better — that is a lead worth following. Judging it against the run's best would
> declare it dead on arrival and cut off exactly the exploration the generous
> run-level patience exists to protect.

## 5. Two layers: told, then stopped

Now the design question, and it is the same one as `train_tabular_model` writing
its own leaderboard row.

Putting "stop when gains stall" in the system prompt is **unfalsifiable**. A model
that keeps going has not violated anything you can point to, and you would never
know the guidance failed — exactly like asking an agent to log its own
experiments, where a forgotten row and an experiment that never happened look
identical from outside.

So the harness owns it. But a bare refusal produces a confused agent — it calls
the tool, gets rejected, does not know why, tries again. Hence two layers:

- **Advisory**: every training result carries a `note` saying where the run
  stands. A
  cooperative model reads it and finalizes on its own, never getting refused.
- **Structural**: past the patience budget, the harness stops accepting training
  calls at all.

Guidance for the cooperative case, structure for the rest.

In [7]:
print("fresh best :", advisory(evaluate(observed[:4], "logloss", policy)))
print()
print("stalling   :", advisory(evaluate(observed[:6], "logloss", policy)))
print()
print("over       :", advisory(evaluate(observed, "logloss", policy)))

fresh best : That is the best result so far. 21 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.

stalling   : 2 experiment(s) since the last real improvement. 19 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.

over       : 7 experiment(s) since the last real improvement. 14 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.


## 6. Watch it happen

A tiny dataset and a deliberately impatient policy, driven by hand.

In [8]:
import os, tempfile
import numpy as np
from tabular_agent.harness.session import RunSession
from tabular_agent.harness.tools import inspect_data, train_tabular_model, finalize_run
from tabular_agent.harness import journal
from tabular_agent.taskspec import TaskSpec

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(size=n)
frame = pd.DataFrame({
    "x1": x1,
    "city": rng.choice(["berlin", "hamburg", "koeln"], size=n),
    "y": (x1 + rng.normal(scale=0.4, size=n) > 0).astype(int),
})
workdir = tempfile.mkdtemp(prefix="phase3_step4_")
data_path = os.path.join(workdir, "d.csv")
frame.to_csv(data_path, index=False)

session = RunSession(
    run_dir=os.path.join(workdir, "run"),
    data_path=data_path,
    spec=TaskSpec(target="y", n_splits=3),
    stop_policy=StopPolicy(table_patience=2, max_refusals=2),
)
inspect_data(session)
train_tabular_model(session, model_type="baseline")

for i in range(3):
    out = train_tabular_model(session, model_type="lgbm")
    print(f"call {i + 1}: trained={out['trained']}", "|", out["note"][:100])

call 1: trained=True | That is the best result so far. 23 experiments remain in this run's budget; the run ends after 10 in


call 2: trained=True | 1 experiment(s) on this table without an improvement; 1 more and the harness will stop training on i


call 3: trained=True | This table is used up: 2 experiments on it without an improvement, so the harness will not train on 


The same model on the same table over and over, so nothing after the first can be
an improvement. The notes escalate — best result, then a warning, then a refusal
on the fourth call.

The ordering is worth keeping straight: the advisory describes the run *after*
the experiment you just did, while the refusal is decided *before* the next one
starts. So a model always gets told once before it gets refused.

That refusal is a **return value**, not an exception — a real decision, not a
stylistic one. Raising would consume the tool's retry budget (`TOOL_RETRIES = 3`),
and exhausting that ends the run and throws away every completed experiment.
Being asked to train once more is not a malformed argument; it is a model that has
not yet read the news. It should cost nothing.

And note *what* is refused:

In [9]:
refused = train_tabular_model(session, model_type="lgbm")
print(json.dumps({k: v for k, v in refused.items() if k != "note"}, indent=2))
print()
print(refused["note"])
print()
print("leaderboard untouched:", [r["id"] for r in journal.load(session.run_dir)["leaderboard"]])

{
  "trained": false,
  "table": "the raw input table",
  "table_exhausted": true,
  "reason": "2 experiments on the raw input table in a row produced no real improvement, so it is only the model and its hyper-parameters changing now. That is the cheapest experiment available and the least informative one",
  "next": "transform_features or fit_encoding, then train on the new table"
}

This table is used up: 2 experiments on it without an improvement, so the harness will not train on it again. Build new features with transform_features or fit_encoding and train on the table that produces. 21 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.

leaderboard untouched: ['e1', 'e2', 'e3', 'e4']


The **table**, not the run. So do what it says — build new features — and
training resumes immediately, with no penalty and no memory of having been turned
away.

In [10]:
from tabular_agent.harness.tools import fit_encoding

enc = fit_encoding(session, ["city"], method="frequency")
resumed = train_tabular_model(session, table=enc["table"], model_type="lgbm")
print("trained:", resumed["trained"], "| experiment:", resumed["experiment_id"])
print(resumed["note"])

trained: True | experiment: e5
3 experiment(s) since the last real improvement. 20 experiments remain in this run's budget; the run ends after 10 in a row without an improvement.


That is the whole idea. The harness did not decide the run was over. It decided
*one table* was over — a much smaller and much safer claim — and the run carries
on for as long as the agent keeps having ideas.

## 7. What if it will not take no for an answer

A redirect only works on someone who takes it. A model that ignores the message
and keeps asking for the same dead table would loop until the context died — and
the run would end with no bundle, which is the one outcome that makes the whole
run worthless.

So repeats of a refused table stop being read as "needs telling" and start being
read as "stuck". The harness escalates them to the run level, and there, after
`max_refusals`, it packages the run **itself**. Only once the deliverable is
safely on disk does it start raising.

In [11]:
# Keep demanding the table it was already turned away from.
for i in range(4):
    out = train_tabular_model(session, model_type="lgbm")
    print(f"call {i + 1}: finalized={out.get('harness_finalized', False)} | {out['reason'][:70]}")
    if out.get("harness_finalized"):
        print("\nbundle:", out["bundle_dir"], "|", out["note"])
        break

call 1: finalized=False | the same exhausted table was requested repeatedly
call 2: finalized=True | the same exhausted table was requested repeatedly

bundle: bundle | Asked to train after the run was over, so the harness packaged the run itself. Nothing further will be trained. Report what you found and stop.


In [12]:
try:
    train_tabular_model(session, model_type="lgbm")
except RuntimeError as e:
    print("RuntimeError:", e)

RuntimeError: this run is finished: the harness has already packaged it and will not train again (the same exhausted table was requested repeatedly)


Read the escalation as three different judgements about the same call:

| Call | Response | Why |
|---|---|---|
| table is used up | refuse *the table*, name the next move | it just needs telling; the run is fine |
| same dead table again | escalate to the run level | this is not a redirect any more, it is a loop |
| `max_refusals`th | harness finalizes, returns the bundle | the deliverable matters more than being obeyed |
| after that | `RuntimeError` | the run is over; there is nothing left to protect |

**Check yourself:** `finalize_run` auto-trains a baseline if the run never did.
That is a call to `train_tabular_model`, made *by the harness*, while the run is
stalled. Why does the stop rule not refuse it?

> Because it is the harness training on its own behalf, not the agent asking for
> one more shot — hence `enforce_stop=False` on that call. Refusing it would
> leave the report unable to answer "did this beat doing nothing", which is the
> one question it must always be able to answer. Worth noticing that the rule had
> to be given an exception the moment it met the *other* rule; that is normal, and
> it is why the exception is a named argument with a comment rather than a
> coincidence.

## 8. Did it cost anything?

The step-3 run, uncapped, did 11 experiments and finished at `0.2718`. The same
prompt and model with the stop rule in place:

| | experiments | best | refusals |
|---|---|---|---|
| uncapped (step 3) | 11 | 0.2718 | — |
| with the stop rule | **8** | 0.2743 | 1 |

Three fewer experiments for a difference of `0.0025` — against a fold spread of
`0.0209`. In other words, it gave up nothing measurable, and the agent complied
with the first refusal rather than fighting it.

**Check yourself:** the capped run scored slightly *worse*. Is that a cost?

> Not one you can defend. `0.0025` is an eighth of the fold spread — the two
> numbers are the same measurement. If you treat that gap as real you have
> re-adopted the exact belief the stop rule exists to reject, and you will always
> be able to justify one more experiment.

## 9. What the run does not entitle you to conclude

A stop rule decides when to stop. It says nothing about whether you can *explain*
the result — and a run that stops at the right time can still leave you unable to
say what worked.

Here is the family tree from the live run above. A "table" is just a file: the
dataset plus some extra columns. Each one is `(an earlier file) + (new columns)`,
and `t3`/`t4` branching off `t2` means both start from the same 12 columns and
then differ by one step:

```
apartments.csv  (7 cols)  -- the original
├── t1   raw + "how common is each city / date"        9 cols  -> 0.6447
├── t2   raw + date split into year/month/day/...     12 cols  -> 0.4743
│   ├── t3   t2 + target-encoded city                 14 cols  -> 0.4715
│   └── t4   t2 + frequency-encoded city              14 cols  -> 0.4637, 0.4027, 0.3160
└── t5   raw + date parts + ratios, logs, squares     23 cols  -> never trained
    └── t6   t5 + frequency-encoded city              25 cols  -> 0.2741  (best)
```

Look at the bottom branch. `t5` was built and then immediately extended into `t6`,
and only `t6` was ever trained. So the chain reads `0.3160 -> (nothing) -> 0.2741`,
and the jump cannot be split between what `t5` added and what `t6` added. There
is no measurement in between.

**In this particular run you can still recover it, by luck of the tree.** `t4`
already contains the frequency encoding and scored `0.3160`; `t6` is `t4`'s idea
plus `t5`'s extra columns and scored `0.2741`. Same model type both times, so the
pile of ratios and logs is worth about `0.042` — twice the fold spread, so real.
But nothing *arranged* that control. Had the agent gone `t2 -> t5 -> t6` without
ever building `t4`, the attribution would simply be gone.

In [13]:
from tabular_agent.harness import journal as journal_mod

run = {
    "task": {"metric": "logloss"},
    "tables": [
        {"id": "t5", "path": "tables/t5.parquet", "source_table": "apartments.csv",
         "method": "transform", "new_columns": ["log_price", "price_sq", "price_per_area"]},
        {"id": "t6", "path": "tables/t6.parquet", "source_table": "tables/t5.parquet",
         "method": "frequency", "new_columns": ["city_freq"]},
    ],
    "leaderboard": [
        {"id": "e9", "model_type": "catboost", "cv_score": 0.2741,
         "source_table": "tables/t6.parquet"},
    ],
}
for step in journal_mod.attribution(run, "tables/t6.parquet"):
    print(f"{step['table']:20} measured={step['measured']!s:5} "
          f"experiments={step['n_experiments']}  added={step['added']}")

tables/t6.parquet    measured=True  experiments=1  added=['city_freq']
tables/t5.parquet    measured=False experiments=0  added=['log_price', 'price_sq', 'price_per_area']


`finalize_run` puts that in the report, and returns `unmeasured_tables` so the
agent sees it before writing its summary — which matters, because the summary is
exactly where a model says "the ratio features were the breakthrough" about a
table nobody scored.

In [14]:
from tabular_agent.report import build_report

text = build_report(
    task_type="binary", metric="logloss", selection_score=0.2741,
    fold_metrics=[0.27, 0.28, 0.26, 0.29], baseline_score=0.6903,
    model_types=["catboost"], feature_names=["..."],
    attribution=journal_mod.attribution(run, "tables/t6.parquet"),
)
print(text.split("## How the winning")[1])

 feature set was built

- `tables/t5.parquet` — transform: log_price, price_sq, price_per_area — **never trained on**
- `tables/t6.parquet` — frequency: city_freq — best 0.2741 over 1 experiment(s)

> **Attribution gap:** `tables/t5.parquet` was built and then extended without ever being trained on. The winning score therefore cannot be split between the columns that table added and the ones added after it — there is no measurement in between. The model is valid; the explanation of *which features earned the score* is not available from this run.


Note what this deliberately is **not**: it does not refuse to build on an
untrained table. That was the alternative — force a measurement at every link, so
attribution is always possible — and it was rejected for the same reason the
run-level patience is generous. Constraining the search is expensive; saying out
loud what the search did not measure is free.

It is the same move as reporting `fold_spread` next to the mean. The harness does
not stop you believing a number. It makes sure you can see what the number does
not cover.

**Check yourself:** the bundle ships all 25 columns of `t6`, including
`price_sq` and `is_studio`. Does the attribution gap mean the *model* is
suspect?

> No. The model was trained and cross-validated on exactly those 25 columns and
> scored `0.2741` doing it, and the bundle reproduces those columns from raw
> input. What is unavailable is the *explanation* — which of the 25 mattered.
> Confusing "I cannot attribute this" with "this is wrong" would be its own
> error, and it is why the report says the model is valid in the same breath.

## 10. Feedback the agent can actually act on

Everything so far is about *reporting*. This section is about the agent's own
iteration, which is a different problem — and the fix for one is not the fix for
the other.

Consider what a training result told the agent before this section: a `cv_score`,
a `fold_spread`, an experiment id. It had just added sixteen columns and got back
one number. There is no channel through which it can learn that `price_sq` did
nothing, so it keeps generating candidates blind. That is why the kitchen-sink
transform happens.

The obvious fix — per-column feature importance — is a trap twice over.

Section 9 gives the first reason: a feature that only pays off next to another
scores nothing alone, so per-column feedback teaches the agent to throw away
whichever half of an interaction arrived first.

The second is that importance answers a different question. It decomposes *one
fitted model's attention*; what the agent needs is a counterfactual — what the
score would be if these columns had never been built. Those come apart routinely:
a column the trees split on constantly can be worth nothing once removed, because
a correlated column carries the same signal, and a binary column offering a single
split point can carry the entire gain.

So `explain_table` measures the counterfactual directly — fit once with every
column, then once more per contributing table with that table's columns held out.
Credit is grouped by table because one tool call writes exactly one table, which
makes the grouping the unit of *decision*: you cannot add half a call.


In [15]:
from tabular_agent.harness.tools import transform_features as transform_tool

# A dataset where one idea is real (the listing date) and one is not (squares).
rng2 = np.random.default_rng(7)
m = 600
listed = pd.to_datetime("2023-01-01") + pd.to_timedelta(rng2.integers(0, 730, size=m), unit="D")
price = rng2.uniform(80_000, 900_000, size=m).round(0)
area = rng2.uniform(30, 160, size=m).round(1)
season = np.where(np.isin(listed.month, [3, 4, 5]), 1.6, -0.8)
flats = pd.DataFrame({
    "price": price, "area": area, "listed_at": listed.strftime("%Y-%m-%d"),
    "sold_fast": ((season + rng2.normal(0, 0.6, size=m)) > 0).astype(int),
})
flats_path = os.path.join(workdir, "flats.csv")
flats.to_csv(flats_path, index=False)

feature_session = RunSession(
    run_dir=os.path.join(workdir, "features"),
    data_path=flats_path,
    spec=TaskSpec(target="sold_fast", n_splits=4),
)
inspect_data(feature_session)

dates = '''
import pandas as pd
def transform_features(df):
    df = df.copy()
    d = pd.to_datetime(df["listed_at"])
    df["m"] = d.dt.month
    df["dow"] = d.dt.dayofweek
    return df
'''
squares = '''
def transform_features(df):
    df = df.copy()
    df["price_sq"] = df["price"] ** 2
    df["area_sq"] = df["area"] ** 2
    return df
'''
print("ready")

ready


In [16]:
from tabular_agent.harness.tools import explain_table

b1 = transform_tool(feature_session, dates)
train_tabular_model(feature_session, table=b1["table"], model_type="lgbm")

b2 = transform_tool(feature_session, squares, table=b1["table"])
out = train_tabular_model(feature_session, table=b2["table"], model_type="lgbm")
print("cv:", round(out["cv_score"], 4), "| vs_source_table:", out["vs_source_table"])

# Which of those calls earned it takes refits, so it is a separate tool.
credit = explain_table(feature_session, table=b2["table"], model_type="lgbm")
print()
for r in credit["credit_by_table"]:
    print(f"  {r['table']:22} {r['method'] or 'raw':10} "
          f"n={r['n_columns']:2} contribution={r['contribution']}")


cv: 0.5134 | vs_source_table: 0.0



  the raw input table    raw        n= 2 contribution=-0.001401
  tables/t1.parquet      transform  n= 2 contribution=0.353632
  tables/t2.parquet      transform  n= 2 contribution=0.0


```
cv: 0.5134 | vs_source_table: 0.0

  the raw input table    raw        n= 2 contribution=-0.001401
  tables/t1.parquet      transform  n= 2 contribution=0.353632
  tables/t2.parquet      transform  n= 2 contribution=0.0
```

The date call carries the whole model: hold its two columns out and the score
loses `0.354`. The squares call is free to remove — `0.0` — and added nothing over
its parent either. That is a legible next move, which one scalar never was.

Now look at `price` and `area`, because they are the reason this is measured by
refitting instead of read off the fitted model. Split-count importance scores them
at **0.702** — they are continuous, so a tree finds somewhere to split on them all
day. Their actual contribution is **-0.001**: hold them out and the score does not
move. `sold_fast` was generated from the listing month alone, so the counterfactual
is right and the importance number is wrong — not imprecise, wrong about which
columns matter.

Two details doing more work than they look:

**`vs_source_table` only exists when the same fit was run on the source table.**
Section 9 rejected *forcing* a measurement at every link. This gets the same
behaviour by making it worth something: train a table before extending it and you
get a marginal number; skip that, or change model and features in one step, and you
get `null` plus a line saying what would produce it. Rewarding the good path rather
than mandating it — the same shape as the advisory that got the agent off `t6`
without ever refusing it.

**A zero means "not needed here", not "useless".** Leave-one-out measures each
table's worth *given the rest*, so two calls carrying the same signal would each
score ~0, each covered by the other. Nothing prunes automatically, and there is no
tool that drops a column.


### One call, one idea

Credit per table is only useful if each call is one hypothesis. The `t5` from the
live run was not — sixteen columns spanning date parts, price ratios, logs, squares
and one-hots. Attributing to it says "these sixteen things were worth 0.042", which
is four ideas in a trenchcoat and unactionable.

So the system prompt steers towards one idea per call. What it no longer has to say
is *keep interacting features together* — and the reason is worth following, because
an earlier version of this notebook argued the opposite.

The worry was a real tension: one feature per call gives clean attribution but was
thought to hide interactions, while a big batch discovers them and makes credit
unrecoverable. The first half of that is wrong under `explain_table`. Tables chain,
so a call built on top of an earlier one puts both sets of columns in the same fit
and the model can find the joint effect either way. And leave-one-out credits both
halves: hold out either and the pair breaks, so both score high.

Verified on a pure XOR whose halves were added in *separate* calls — each scores
about `+0.82` (`test_both_halves_of_an_interaction_are_credited_across_separate_calls`).
So the guidance is simply one idea per call, plus one warning that does still apply:
**a call that scores nothing on its own is not disproof** — its partner may not
exist yet.

This remains a **steer, not a rule**. Nothing refuses a mixed transform, and it
would be wrong to: sometimes the kitchen sink is the right opening move precisely
because it catches interactions you had not thought of.

**Check yourself:** the agent trains on `t2`, sees the squares call at `0.0`, and
wants to remove those two columns. Which tool does that?

> None — and that is deliberate. There is no tool that drops columns. Tables are
> immutable, so the move is to stop building on `t2` and extend `t1` instead; `t2`
> stays on disk, costing nothing, in case the idea earns its place later.


## 11. What this does not do

Worth being precise about, because a stop rule is easy to over-read:

- It does **not** know whether the *ideas* are exhausted, only whether the
  measurements have stopped moving. "This table is used up" is a claim about a
  table, not about the dataset — there may be an obvious feature nobody tried.
- It is **evadable**, as section 4 showed: alternating between two dead tables
  keeps both per-table counters under the limit. The generous run-level patience
  catches that eventually, several experiments later.
- It does **not** protect against a bad metric or a leaky feature. A run leaking
  the target converges fast and stops early, pleased with itself.
- `max_experiments` is a **budget**, not a judgement. A run that hits it may have
  been mid-climb, and the report will not say so.
- It cannot tell "tuning too early" from "tuning at the end, on purpose", which is
  the one time tuning is the right move. It just makes tuning expensive to do at
  length on any single table — which happens to push it towards the end, but by
  accident rather than by design.

**Next:** `blend_models` and `predict` are the two tools still missing, then §19
compaction — the run journal grows without bound, and a long run will eventually
outlive the context it is being read into.